## This project will primarily use Python and SQL, with Pandas and DuckDB as the main libraries for data manipulation, transformation, and analysis.

Pandas provides flexible data handling for exploration and cleaning, while DuckDB allows efficient SQL queries directly on Parquet files.
This lightweight stack is sufficient for the dataset size.

## This notebook will be used for the exploratory data analysis, this notebook leads to no technical results.

In [1]:
#Run this cell to import necessary libraries and set up data paths for the analysis. Do not edit this cell.
import pandas as pd
from pathlib import Path
import duckdb
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

data_path = Path("../data/raw")
visitors_path = (data_path / "visitors.parquet").as_posix()
sessions_path = (data_path / "sessions.parquet").as_posix()
events_path = (data_path / "questionnaire_events.parquet").as_posix()
answers_path = (data_path / "questionnaire_answers.parquet").as_posix()
questions_path = (data_path / "questionnaire_questions.parquet").as_posix()
outcomes_path = (data_path / "questionnaire_outcomes.parquet").as_posix()

tables = {
    "visitors": pd.read_parquet(visitors_path),
    "sessions": pd.read_parquet(sessions_path),
    "questionnaire_questions": pd.read_parquet(questions_path),
    "questionnaire_events": pd.read_parquet(events_path),
    "questionnaire_answers": pd.read_parquet(answers_path),
    "questionnaire_outcomes": pd.read_parquet(outcomes_path)
}


↓ We started by verifying that the data could be read correctly and checking for orphaned records across the related tables.

In [2]:
##Test code to read all parquet files in the data directory

for file in data_path.glob("*.parquet"):
    df = pd.read_parquet(file)
    print(file.name, df.shape)
    display(df.head())

questionnaire_answers.parquet (68460, 7)


,answer_id,session_id,visitor_id,question_id,question_number,answer_value,answered_at
0,ANS_000000001,SES_00000001,VIS_00016128,diabetes_q1,1,no,2026-06-01 00:02:35.588484793+00:00
1,ANS_000000002,SES_00000002,VIS_00012147,diabetes_q1,1,no,2026-06-01 00:09:23.966499272+00:00
2,ANS_000000003,SES_00000002,VIS_00012147,diabetes_q2,2,no,2026-06-01 00:09:48.773896500+00:00
3,ANS_000000004,SES_00000003,VIS_00018597,diabetes_q1,1,no,2026-06-01 00:11:09.794015705+00:00
4,ANS_000000005,SES_00000003,VIS_00018597,diabetes_q2,2,no,2026-06-01 00:11:33.897800263+00:00


questionnaire_events.parquet (195816, 6)


,event_id,session_id,visitor_id,event_timestamp,event_type,question_number
0,EVT_000000001,SES_00000001,VIS_00016128,2026-06-01 00:01:59.429044608+00:00,landing_view,<NA>
1,EVT_000000002,SES_00000001,VIS_00016128,2026-06-01 00:02:07.868799740+00:00,questionnaire_start,<NA>
2,EVT_000000003,SES_00000001,VIS_00016128,2026-06-01 00:02:11.208955362+00:00,question_view,1
3,EVT_000000004,SES_00000001,VIS_00016128,2026-06-01 00:02:35.588484793+00:00,question_answer,1
4,EVT_000000005,SES_00000001,VIS_00016128,2026-06-01 00:02:36.940089191+00:00,question_view,2


questionnaire_outcomes.parquet (11357, 4)


...

,question_id,question_number,question_text,response_type,allowed_answers
0,diabetes_q1,1,Have you ever been told by a healthcare professional that you have Type 2 Diabetes?,single_choice,"[""yes"", ""no"", ""prefer_not_to_say""]"
1,diabetes_q2,2,"During the last 3 months, have you frequently experienced unusual thirst or frequent urination?",single_choice,"[""yes"", ""no"", ""not_sure""]"
2,diabetes_q3,3,Has a parent or sibling been diagnosed with Type 2 Diabetes?,single_choice,"[""yes"", ""no"", ""not_sure""]"
3,diabetes_q4,4,How often do you usually do at least 30 minutes of physical activity?,single_choice,"[""5_or_more_days_per_week"", ""3_to_4_days_per_week"", ""1_to_2_days_per_week"", ""rarely_or_never""]"
4,diabetes_q5,5,"Have you ever been told that your blood sugar was higher than normal, without receiving a Type 2 Diabetes diagnosis?",single_choice,"[""yes"", ""no"", ""not_sure""]"


sessions.parquet (26000, 8)


,session_id,visitor_id,session_started_at,acquisition_source,campaign_name,device_type,landing_page,is_returning_visitor
0,SES_00000001,VIS_00016128,2026-06-01 00:01:59.429044608+00:00,social,diabetes_social_awareness,desktop,/health/diabetes-awareness,False
1,SES_00000002,VIS_00012147,2026-06-01 00:07:54.468178789+00:00,google_organic,NaN,desktop,/health/diabetes-awareness,False
2,SES_00000003,VIS_00018597,2026-06-01 00:10:06.909833040+00:00,google_ads,diabetes_awareness_general,desktop,/health/diabetes-awareness,False
3,SES_00000004,VIS_00006671,2026-06-01 00:14:47.814240001+00:00,google_organic,NaN,mobile,/health/diabetes-awareness,False
4,SES_00000005,VIS_00018327,2026-06-01 00:14:55.885097705+00:00,chatgpt,NaN,mobile,/health/diabetes-awareness,False


visitors.parquet (20000, 5)


,visitor_id,birth_year,gender,region,first_seen_at
0,VIS_00000001,2008,male,Bretagne,2026-08-02 04:29:33.074952518+00:00
1,VIS_00000002,1946,female,Hauts-de-France,2026-07-25 11:59:57.752213928+00:00
2,VIS_00000003,<NA>,female,Pays de la Loire,2026-06-22 12:37:47.114315535+00:00
3,VIS_00000004,1989,other,Provence-Alpes-Côte d'Azur,2026-07-12 02:29:05.408274007+00:00
4,VIS_00000005,1989,female,Hauts-de-France,2026-07-10 05:33:03.390070397+00:00


In [3]:
## Test code to read all parquet files in the data directory

duckdb.sql(f"""
SELECT *
FROM read_parquet('{data_path.as_posix()}/*.parquet')
LIMIT 5
""").show()

┌───────────────┬──────────────┬──────────────┬─────────────┬─────────────────┬──────────────┬───────────────────────────────┐
│   answer_id   │  session_id  │  visitor_id  │ question_id │ question_number │ answer_value │          answered_at          │
│    varchar    │   varchar    │   varchar    │   varchar   │      int64      │   varchar    │   timestamp with time zone    │
├───────────────┼──────────────┼──────────────┼─────────────┼─────────────────┼──────────────┼───────────────────────────────┤
│ ANS_000000001 │ SES_00000001 │ VIS_00016128 │ diabetes_q1 │               1 │ no           │ 2026-06-01 02:02:35.588484+02 │
│ ANS_000000002 │ SES_00000002 │ VIS_00012147 │ diabetes_q1 │               1 │ no           │ 2026-06-01 02:09:23.966499+02 │
│ ANS_000000003 │ SES_00000002 │ VIS_00012147 │ diabetes_q2 │               2 │ no           │ 2026-06-01 02:09:48.773896+02 │
│ ANS_000000004 │ SES_00000003 │ VIS_00018597 │ diabetes_q1 │               1 │ no           │ 2026-06-01 02:11

↓ No orphaned records were found, and all datasets were successfully read. We can now proceed with the exploratory data analysis.

The results soon provided will not only be used for a future data clean but also to have a clear perspective over future analytical paths.

In [4]:
## Check for orphaned records in child tables that do not have a corresponding parent record.

checks = [
    ("sessions", sessions_path, "visitors", visitors_path, "visitor_id"),
    ("questionnaire_events", events_path, "sessions", sessions_path, "session_id"),
    ("questionnaire_answers", answers_path, "sessions", sessions_path, "session_id"),
    ("questionnaire_answers", answers_path, "questionnaire_questions", questions_path, "question_id"),
    ("questionnaire_outcomes", outcomes_path, "sessions", sessions_path, "session_id")
]

for child, child_path, parent, parent_path, key in checks:
    n = duckdb.sql(f"""
        SELECT COUNT(*)
        FROM read_parquet('{child_path}') c
        LEFT JOIN read_parquet('{parent_path}') p USING ({key})
        WHERE p.{key} IS NULL
    """).fetchone()[0]

    print(f"{child} → {parent}: {n} orphelins")

sessions → visitors: 0 orphelins
questionnaire_events → sessions: 0 orphelins
questionnaire_answers → sessions: 0 orphelins
questionnaire_answers → questionnaire_questions: 0 orphelins
questionnaire_outcomes → sessions: 0 orphelins


↓ No duplicated rows in the provided datasets, and no missing relevant IDs.

In [5]:
# Check for duplicates and missing relevant IDs in each table.

key_columns = {
    "visitors": ["visitor_id"],
    "sessions": ["session_id", "visitor_id"],
    "questionnaire_questions": ["question_id"],
    "questionnaire_events": ["event_id", "session_id", "visitor_id"],
    "questionnaire_answers": ["answer_id", "session_id", "visitor_id", "question_id"],
    "questionnaire_outcomes": ["session_id", "visitor_id"]
}

for name, df in tables.items():
    duplicates = df.duplicated().sum()
    missing_ids = df[key_columns[name]].isna().any(axis=1).sum()

    print(f"{name}:")
    print(f"  Exact duplicates: {duplicates}")
    print(f"  Rows with missing relevant IDs: {missing_ids}\n")

visitors:
  Exact duplicates: 0
  Rows with missing relevant IDs: 0

sessions:
  Exact duplicates: 0
  Rows with missing relevant IDs: 0

questionnaire_questions:
  Exact duplicates: 0
  Rows with missing relevant IDs: 0

questionnaire_events:
  Exact duplicates: 0
  Rows with missing relevant IDs: 0

questionnaire_answers:
  Exact duplicates: 0
  Rows with missing relevant IDs: 0

questionnaire_outcomes:
  Exact duplicates: 0
  Rows with missing relevant IDs: 0



↓ No major inconsistencies were identified in the sessions related data.


In [6]:
#Check for the number of completed questionnaires and the number of outcomes to see if they match.
duckdb.sql(f"""
SELECT
    (SELECT COUNT(*)
     FROM read_parquet('{events_path}')
     WHERE event_type = 'questionnaire_complete') AS completed_questionnaires,

    (SELECT COUNT(*)
     FROM read_parquet('{outcomes_path}')) AS outcomes
""").df()

,completed_questionnaires,outcomes
0,11357,11357


In [7]:
#Check if a session has multiple outcomes, which should not happen.
duckdb.sql(f"""
SELECT COUNT(*) AS sessions_with_multiple_outcomes
FROM (
    SELECT session_id
    FROM read_parquet('{outcomes_path}')
    GROUP BY session_id
    HAVING COUNT(*) > 1
)
""").df()

,sessions_with_multiple_outcomes
0,0


↓ After several checks on the sessions and outcomes tables, we identified visitors who completed multiple questionnaires and received different outcomes across sessions. This may reflect inconsistent self-reported data, repeated testing behavior, or genuine changes over time. We also identified visitors whose outcomes followed a coherent progression, such as no_current_indication → possible_risk → declared_diagnosed, suggesting that not all outcome changes should be treated as data-quality issues.

In [8]:
#check for visitors with multiple outcomes which is not expected in the data. This is a data quality issue that needs to be addressed
duckdb.sql(f"""
SELECT
    visitor_id,
    COUNT(*) AS outcome_count
FROM read_parquet('{outcomes_path}')
GROUP BY visitor_id
HAVING COUNT(*) > 1
ORDER BY outcome_count DESC
""").df()

,visitor_id,outcome_count
0,VIS_00005172,3
1,VIS_00007088,3
2,VIS_00006054,3
3,VIS_00011358,3
4,VIS_00006369,3
...,...,...
1498,VIS_00015293,2
1499,VIS_00011163,2
1500,VIS_00014243,2
1501,VIS_00002131,2


In [9]:
#check for visitors with multiple completed sessions.
#They may have participated in multiple campaigns but they also may have provided different answers to the same questrinnaires.
#We must check if the outcomes check and if not it must be adressed as a data quality issue.
duckdb.sql(f"""
SELECT
    o.visitor_id,
    COUNT(DISTINCT o.session_id) AS completed_sessions,
    LIST(DISTINCT s.acquisition_source) AS sources,
    LIST(DISTINCT s.campaign_name) AS campaigns
FROM read_parquet('{outcomes_path}') o
JOIN read_parquet('{sessions_path}') s
    ON o.session_id = s.session_id
GROUP BY o.visitor_id
HAVING COUNT(DISTINCT o.session_id) > 1
ORDER BY completed_sessions DESC
""").df()

,visitor_id,completed_sessions,sources,campaigns
0,VIS_00005348,3,"[email, direct, social]","[diabetes_retargeting, None, diabetes_awareness_general]"
1,VIS_00003642,3,"[chatgpt, google_ads]","[diabetes_retargeting, None]"
2,VIS_00001385,3,"[google, email, google_ads]","[None, diabetes_awareness_general, diabetes_45plus]"
3,VIS_00006634,3,"[email, social, google_organic]","[diabetes_awareness_general, None, diabetes_retargeting]"
4,VIS_00011436,3,"[google_organic, AI Assistant, chatgpt]",[None]
...,...,...,...,...
1498,VIS_00009548,2,"[direct, google_organic]","[None, diabetes_retargeting]"
1499,VIS_00015744,2,[referral],[None]
1500,VIS_00010042,2,"[google_organic, direct]","[diabetes_retargeting, None]"
1501,VIS_00008511,2,"[social, email]",[diabetes_retargeting]


In [10]:
#check for the total number of completed sessions across all visitors with multiple completed sessions.
duckdb.sql(f"""
SELECT SUM(completed_sessions) AS total_completed_sessions
FROM (
    SELECT visitor_id, COUNT(*) AS completed_sessions
    FROM read_parquet('{outcomes_path}')
    GROUP BY visitor_id
    HAVING COUNT(*) > 1
)
""").df()

,total_completed_sessions
0,3390.0


In [11]:
#Check for visitors with multiple outcomes but reccurrent outcomes, which may be expected in the data.
#Visitors may also have participated in multiple campaigns.
duckdb.sql(f"""
SELECT
    o.visitor_id,
    COUNT(*) AS outcome_count,
    LIST(DISTINCT o.outcome_category) AS outcomes,
    LIST(DISTINCT s.campaign_name) AS campaigns
FROM read_parquet('{outcomes_path}') o
JOIN read_parquet('{sessions_path}') s
    ON o.session_id = s.session_id
GROUP BY o.visitor_id
HAVING COUNT(*) > 1
   AND COUNT(DISTINCT o.outcome_category) = 1
ORDER BY outcome_count DESC
""").df()

,visitor_id,outcome_count,outcomes,campaigns
0,VIS_00015683,3,[no_current_indication],"[diabetes_retargeting, None]"
1,VIS_00008030,3,[no_current_indication],"[diabetes_45plus, None]"
2,VIS_00014235,3,[no_current_indication],"[diabetes_45plus, None, diabetes_awareness_general]"
3,VIS_00017599,3,[no_current_indication],"[diabetes_social_awareness, diabetes_awareness_general]"
4,VIS_00019801,3,[no_current_indication],"[None, diabetes_retargeting]"
...,...,...,...,...
820,VIS_00003173,2,[no_current_indication],"[None, diabetes_retargeting]"
821,VIS_00010428,2,[no_current_indication],"[None, diabetes_retargeting]"
822,VIS_00007325,2,[no_current_indication],"[diabetes_retargeting, diabetes_45plus]"
823,VIS_00000255,2,[no_current_indication],"[None, diabetes_retargeting]"


In [12]:
#Check for visitors with multiple distinct outcomes, which is not expected in the data. This is a data quality issue that needs to be addressed.
#Data may not be coherent in the case where a visitor may have been diagnosed after a first session and then after a second session they may have provided a different answer. Must be adressed.
duckdb.sql(f"""
SELECT
    visitor_id,
    COUNT(DISTINCT outcome_category) AS distinct_outcomes,
    LIST(DISTINCT outcome_category) AS outcomes
FROM read_parquet('{outcomes_path}')
GROUP BY visitor_id
HAVING COUNT(DISTINCT outcome_category) > 1
ORDER BY distinct_outcomes DESC
""").df()

,visitor_id,distinct_outcomes,outcomes
0,VIS_00009106,3,"[possible_risk, declared_diagnosed, no_current_indication]"
1,VIS_00007614,3,"[no_current_indication, declared_diagnosed, possible_risk]"
2,VIS_00007430,3,"[declared_diagnosed, possible_risk, no_current_indication]"
3,VIS_00008107,3,"[no_current_indication, declared_diagnosed, possible_risk]"
4,VIS_00014648,3,"[no_current_indication, declared_diagnosed, possible_risk]"
...,...,...,...
673,VIS_00007097,2,"[possible_risk, no_current_indication]"
674,VIS_00000170,2,"[possible_risk, no_current_indication]"
675,VIS_00010323,2,"[possible_risk, no_current_indication]"
676,VIS_00008054,2,"[possible_risk, no_current_indication]"


In [13]:
#Check for visitors with multiple outcomes but coherent evolution, which may be expected in the data.
duckdb.sql(f"""
WITH ordered AS (
    SELECT *,
        CASE outcome_category
            WHEN 'no_current_indication' THEN 1
            WHEN 'possible_risk' THEN 2
            WHEN 'declared_diagnosed' THEN 3
        END AS level,
        LAG(level) OVER (
            PARTITION BY visitor_id
            ORDER BY completed_at
        ) AS previous_level
    FROM read_parquet('{outcomes_path}')
)

SELECT
    visitor_id,
    LIST(outcome_category ORDER BY completed_at) AS evolution,
    LIST(STRFTIME(completed_at, '%d/%m/%Y') ORDER BY completed_at) AS dates
FROM ordered
GROUP BY visitor_id
HAVING COUNT(DISTINCT outcome_category) > 1
   AND SUM(CASE WHEN level < previous_level THEN 1 ELSE 0 END) = 0
ORDER BY visitor_id
""").df()

,visitor_id,evolution,dates
0,VIS_00000028,"[no_current_indication, declared_diagnosed]","[06/06/2026, 11/07/2026]"
1,VIS_00000041,"[no_current_indication, no_current_indication, possible_risk]","[22/06/2026, 07/07/2026, 22/07/2026]"
2,VIS_00000170,"[no_current_indication, possible_risk]","[14/07/2026, 31/07/2026]"
3,VIS_00000294,"[no_current_indication, possible_risk]","[02/06/2026, 25/06/2026]"
4,VIS_00000418,"[no_current_indication, no_current_indication, possible_risk]","[06/07/2026, 11/07/2026, 25/07/2026]"
...,...,...,...
275,VIS_00019901,"[no_current_indication, no_current_indication, declared_diagnosed]","[02/07/2026, 14/07/2026, 25/07/2026]"
276,VIS_00019973,"[no_current_indication, no_current_indication, declared_diagnosed]","[14/06/2026, 02/07/2026, 24/07/2026]"
277,VIS_00019979,"[no_current_indication, declared_diagnosed]","[11/06/2026, 27/06/2026]"
278,VIS_00019983,"[no_current_indication, declared_diagnosed]","[26/06/2026, 30/07/2026]"


In [14]:
#check for visitors with multiple outcomes but incoherent evolution, which is not expected in the data. This is a data quality issue that needs to be addressed.
duckdb.sql(f"""
WITH x AS (
    SELECT *,
        CASE outcome_category
            WHEN 'no_current_indication' THEN 1
            WHEN 'possible_risk' THEN 2
            WHEN 'declared_diagnosed' THEN 3
        END AS level,
        LAG(level) OVER (PARTITION BY visitor_id ORDER BY completed_at) AS prev
    FROM read_parquet('{outcomes_path}')
)

SELECT visitor_id,
       LIST(outcome_category ORDER BY completed_at) AS evolution,
       LIST(STRFTIME(completed_at, '%d/%m/%Y') ORDER BY completed_at) AS dates
FROM x
GROUP BY visitor_id
HAVING SUM(CASE WHEN prev = 3 AND level < 3 THEN 1 ELSE 0 END) > 0
ORDER BY visitor_id
""").df()

,visitor_id,evolution,dates
0,VIS_00000204,"[possible_risk, declared_diagnosed, possible_risk]","[05/07/2026, 09/07/2026, 29/07/2026]"
1,VIS_00000222,"[declared_diagnosed, no_current_indication]","[15/07/2026, 26/07/2026]"
2,VIS_00000384,"[declared_diagnosed, no_current_indication, no_current_indication]","[04/07/2026, 11/07/2026, 23/07/2026]"
3,VIS_00000588,"[declared_diagnosed, no_current_indication]","[12/07/2026, 26/07/2026]"
4,VIS_00000732,"[declared_diagnosed, no_current_indication]","[18/07/2026, 24/07/2026]"
...,...,...,...
170,VIS_00019241,"[declared_diagnosed, no_current_indication]","[09/06/2026, 25/07/2026]"
171,VIS_00019333,"[declared_diagnosed, no_current_indication]","[16/07/2026, 31/07/2026]"
172,VIS_00019378,"[declared_diagnosed, possible_risk]","[08/07/2026, 28/07/2026]"
173,VIS_00019630,"[no_current_indication, declared_diagnosed, no_current_indication]","[05/07/2026, 12/07/2026, 23/07/2026]"


↓ The questionnaire and outcome data do not appear to contain any major internal inconsistencies.

In [15]:
#checking for outcomes provided that do not match any completed questionnaire.
duckdb.sql(f"""
SELECT count(*) AS outcomes_without_complete_event
FROM read_parquet('{outcomes_path}') o
LEFT JOIN read_parquet('{events_path}') e
ON o.session_id = e.session_id
AND e.event_type = 'questionnaire_complete'
WHERE e.session_id IS NULL
""").df()

,outcomes_without_complete_event
0,0


In [16]:
#check for questionnaires completed that do not have a corresponding outcome.
duckdb.sql(f"""
SELECT count(*) AS events_complete_without_outcome
FROM read_parquet('{events_path}') e
LEFT JOIN read_parquet('{outcomes_path}') o
ON e.session_id = o.session_id
WHERE e.event_type = 'questionnaire_complete'
AND o.session_id IS NULL
""").df()

,events_complete_without_outcome
0,0


In [17]:
#Check for questionnaires that were completed but did not have all 5 questions answered.
duckdb.sql(f"""
SELECT COUNT(*) AS falsely_completed
FROM (
    SELECT e.session_id
    FROM read_parquet('{events_path}') e
    LEFT JOIN read_parquet('{answers_path}') a
        ON e.session_id = a.session_id
    WHERE e.event_type = 'questionnaire_complete'
    GROUP BY e.session_id
    HAVING COUNT(DISTINCT a.question_number) < 5
)
""").df()

,falsely_completed
0,0


In [18]:
#check for questionnaires that were completed but did not have all 5 questions viewed.
duckdb.sql(f"""
SELECT COUNT(*) AS completed_without_5_views
FROM (
    SELECT session_id
    FROM read_parquet('{events_path}')
    WHERE session_id IN (
        SELECT session_id
        FROM read_parquet('{events_path}')
        WHERE event_type = 'questionnaire_complete'
    )
    GROUP BY session_id
    HAVING COUNT(DISTINCT CASE
        WHEN event_type = 'question_view'
        THEN question_number
    END) < 5
)
""").df()

,completed_without_5_views
0,0


↓ The timestamps appear to be internally consistent, with no major chronological anomalies identified.

In [19]:
#check for events that have started before the session start time, which is not expected in the data.
duckdb.sql(f"""
SELECT COUNT(*) AS events_before_session
FROM read_parquet('{events_path}') e
JOIN read_parquet('{sessions_path}') s USING(session_id)
WHERE e.event_timestamp < s.session_started_at
""").df()

,events_before_session
0,0


In [20]:
#check for questionnaires that got an outcome before their supposed completion, which is not expected in the data.
duckdb.sql(f"""
SELECT COUNT(*) AS answers_after_complete
FROM read_parquet('{answers_path}') a
JOIN read_parquet('{outcomes_path}') o USING(session_id)
WHERE a.answered_at > o.completed_at
""").df()

,answers_after_complete
0,0


In [21]:
#check for events that have a start time after their end time, which is not expected in the data.
duckdb.sql(f"""
SELECT COUNT(*) AS invalid_order
FROM (
    SELECT session_id,
           MIN(CASE WHEN event_type='questionnaire_start' THEN event_timestamp END) AS start_time,
           MIN(CASE WHEN event_type='questionnaire_complete' THEN event_timestamp END) AS end_time
    FROM read_parquet('{events_path}')
    GROUP BY session_id
)
WHERE start_time > end_time
""").df()

,invalid_order
0,0


↓ Gender and region contain non-standardized categorical values, including inconsistent casing, abbreviations, and accent variations, which create duplicate representations of the same category.

In [22]:
#checking for unusual device types in the sessions table, which may indicate data quality issues.
duckdb.sql(f"""
SELECT DISTINCT device_type FROM read_parquet('{sessions_path}');
""").show()

┌─────────────┐
│ device_type │
│   varchar   │
├─────────────┤
│ desktop     │
│ mobile      │
│ tablet      │
└─────────────┘



In [23]:
#checking for false gender values in the sessions table, which may indicate data quality issues.
display(duckdb.sql(f"""
SELECT DISTINCT gender
FROM read_parquet('{visitors_path}')
ORDER BY gender
""").df())

,gender
0,F
1,Female
2,M
3,Male
4,female
5,male
6,other
7,prefer_not_to_say
8,NaN


In [24]:
#checking for incorrect regions in the sessions table, which may indicate data quality issues.
display(duckdb.sql(f"""
SELECT DISTINCT region
FROM read_parquet('{visitors_path}')
ORDER BY region
""").df())

,region
0,Auvergne Rhone Alpes
1,Auvergne-Rhône-Alpes
2,Bretagne
3,Grand Est
4,Hauts-de-France
5,Ile de France
6,Nouvelle-Aquitaine
7,Occitanie
8,Pays de la Loire
9,Provence-Alpes-Côte d'Azur


↓ All the answers provided are allowed.

In [25]:
#Checking if all the questions have the same allowed answers, which is expected in the data.
display(duckdb.sql(f"""
SELECT DISTINCT question_number, answer_value
FROM read_parquet('{answers_path}')
ORDER BY question_number, answer_value
""").df())

,question_number,answer_value
0,1,no
1,1,prefer_not_to_say
2,1,yes
3,2,no
4,2,not_sure
5,2,yes
6,3,no
7,3,not_sure
8,3,yes
9,4,1_to_2_days_per_week


↓ All the provided data respects the time range expected.

In [26]:
#cehcking if all the provided data respects the expected time range of the study, which is from June 1st, 2026 to September 1st, 2026.
duckdb.sql(f"""
SELECT
    (SELECT COUNT(*) FROM read_parquet('{sessions_path}')
     WHERE session_started_at < '2026-06-01' OR session_started_at >= '2026-09-01') AS invalid_sessions,

    (SELECT COUNT(*) FROM read_parquet('{events_path}')
     WHERE event_timestamp < '2026-06-01' OR event_timestamp >= '2026-09-01') AS invalid_events,

    (SELECT COUNT(*) FROM read_parquet('{answers_path}')
     WHERE answered_at < '2026-06-01' OR answered_at >= '2026-09-01') AS invalid_answers,

    (SELECT COUNT(*) FROM read_parquet('{outcomes_path}')
     WHERE completed_at < '2026-06-01' OR completed_at >= '2026-09-01') AS invalid_outcomes
""").df()

,invalid_sessions,invalid_events,invalid_answers,invalid_outcomes
0,0,0,0,0


↓ Birth years appear consistent, ranging from approximately 16 to 96 years old at the time of questionnaire completion.

In [27]:
#checking for extreme birth years in the visitors table, which may indicate data quality issues.
duckdb.sql(f"""
SELECT
    MIN(birth_year) AS min_birth_year,
    MAX(birth_year) AS max_birth_year
FROM read_parquet('{visitors_path}')
""").df()

,min_birth_year,max_birth_year
0,1929,2010


↓ The last checkings on visitor data revealed that many visitors did not provide enough information about themselves which could influence future analysis, i came up with two solutions, we either delete them in the data processing phase or flag them as if they prefered not to provide their personnal data and include them differently on the analysis which is my prefered approach.

In [28]:
#checking for visitors with incomplete data, which is not expected in the data.
duckdb.sql(f"""
SELECT
    COUNT(*) FILTER (WHERE visitor_id IS NULL) AS visitor_id_nulls,
    COUNT(*) FILTER (WHERE birth_year IS NULL) AS birth_year_nulls,
    COUNT(*) FILTER (WHERE gender IS NULL) AS gender_nulls,
    COUNT(*) FILTER (WHERE region IS NULL) AS region_nulls,
    COUNT(*) FILTER (WHERE first_seen_at IS NULL) AS first_seen_at_nulls
FROM read_parquet('{visitors_path}')
""").df()

,visitor_id_nulls,birth_year_nulls,gender_nulls,region_nulls,first_seen_at_nulls
0,0,761,653,804,0


In [29]:
#checking for visitors with all personal data fields incomplete, which is not expected in the data.
duckdb.sql(f"""
SELECT COUNT(*) AS visitors_with_incomplete_data
FROM (
    SELECT
        (birth_year IS NULL)::INT +
        (gender IS NULL)::INT +
        (region IS NULL)::INT AS null_count
    FROM read_parquet('{visitors_path}')
)
WHERE null_count > 2
""").df()

,visitors_with_incomplete_data
0,6


## EDA Conclusion

The exploratory analysis shows that the datasets are structurally consistent: no orphaned records, exact duplicates, missing critical identifiers, major timestamp anomalies, or inconsistencies between completed questionnaires, answers, views, and outcomes were identified.

The main data-quality issue concerns categorical standardization, particularly gender and region, where multiple representations of the same category exist, visitors with little to no personnal information provided should also be adressed seperately.

Visitors completing multiple questionnaires may also receive different outcomes across sessions. These cases should not automatically be treated as errors, as some show coherent evolution over time and may reflect repeated testing or changes in self-reported information.

Overall, the dataset requires limited cleaning, mainly categorical normalization, before proceeding to analytical transformations and deeper business analysis.